# Commuter flow generation

This notebook reproduces the London commuter flow experiments with precomputed Output Area representations. It keeps the complete pipeline visible: configuration, data validation, training, early stopping, evaluation, aggregation, and checkpoint writing.

The default settings run a small CPU-friendly smoke experiment. Set `DEBUG = False` and `USE_MULTIPLE_MODEL_SEEDS = True` to run all five geographic splits and the paper's five model seeds.

## 1. Imports and repository paths

In [1]:
from __future__ import annotations

import ast
import json
import random
import sys
import time
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats
from scipy.special import rel_entr
from torch.utils.data import DataLoader, Dataset

def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'README.md').is_file() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the LE4Mob repository.')

REPOSITORY_ROOT = find_repository_root(Path.cwd())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from le4mob.checkpoint import sha256_file
from mob_tasks.checkpoint import DownstreamCheckpointMetadata, save_downstream_checkpoint
from mob_tasks.commuter_flow import Bilinear, DeepGravity, load_experiment_config, resolve_model_seeds

REPOSITORY_ROOT

PosixPath('/home/xwang/2026/LE4Mob')

## 2. Experiment controls

Change only these values for the usual experiments. `REPRESENTATION` may be `le4mob`, `calliper`, `hand_crafted`, `lda`, `space2vec`, `sppe`, or `urban2vec`. `MODEL_TYPE` may be `deepgravity` or `bilinear`.

In [ ]:
CONFIG_PATH = REPOSITORY_ROOT / 'configs/downstream/commuter_flow/london.yaml'
REPRESENTATION = 'le4mob'
MODEL_TYPE = 'deepgravity'
USE_MULTIPLE_MODEL_SEEDS = False  # if False, only the first model seed (10) will be used for training and evaluation
# If True (which is the setting the paper uses), the model will be trained and evaluated on all model seeds (10, 20, 30, 40, 50) and the results will be averaged across seeds.
DEBUG = True  # Set to False for full training
DEVICE_OVERRIDE = None  # for example: 'cpu' or 'cuda:1'

config = load_experiment_config(CONFIG_PATH)
config['model']['type'] = MODEL_TYPE
config['training']['use_multiple_model_seeds'] = USE_MULTIPLE_MODEL_SEEDS
if REPRESENTATION not in config['data']['representations']:
    raise ValueError(f'Unknown representation: {REPRESENTATION}')

split_ids = config['debug']['split_ids'] if DEBUG else config['data']['split_ids']
model_seeds = [10] if DEBUG else resolve_model_seeds(config['training'])
max_epochs = config['debug']['max_epochs'] if DEBUG else config['training']['max_epochs']
requested_device = DEVICE_OVERRIDE or config['training']['device']
if requested_device.startswith('cuda') and not torch.cuda.is_available():
    requested_device = 'cpu'
device = torch.device(requested_device)

print({
    'representation': REPRESENTATION,
    'model': MODEL_TYPE,
    'splits': split_ids,
    'model_seeds': model_seeds,
    'max_epochs': max_epochs,
    'device': str(device),
    'debug': DEBUG,
})

{'representation': 'le4mob', 'model': 'deepgravity', 'splits': [0, 1, 2, 3, 4], 'model_seeds': [10], 'max_epochs': 100, 'device': 'cuda:0', 'debug': False}


## 3. Load and validate embeddings and flow splits

Pandas reads the compressed CSV files directly. String representations of lists are parsed with `ast.literal_eval`, not `eval`. Output Area identifiers remain strings.

In [3]:
LITERAL_COLUMNS = ('destinations', 'counts', 'distances', 'distance_normed')

def load_embedding_table(path: Path, expected_dimension: int) -> pd.DataFrame:
    table = pd.read_csv(path, dtype={'oa21cd': str})
    required = {'oa21cd', 'embedding'}
    if not required.issubset(table.columns):
        raise ValueError(f'{path} must contain columns {sorted(required)}.')
    if table['oa21cd'].duplicated().any():
        raise ValueError(f'{path} contains duplicate Output Area identifiers.')
    table['embedding'] = table['embedding'].map(ast.literal_eval)
    dimensions = table['embedding'].map(len).unique().tolist()
    if dimensions != [expected_dimension]:
        raise ValueError(f'Expected dimension {expected_dimension}, found {dimensions}.')
    return table.set_index('oa21cd')

def load_flow_split(path: Path, *, limit: int | None = None) -> pd.DataFrame:
    table = pd.read_csv(path, dtype={'origin': str}, nrows=limit)
    required = {'origin', *LITERAL_COLUMNS}
    if not required.issubset(table.columns):
        raise ValueError(f'{path} must contain columns {sorted(required)}.')
    for column in LITERAL_COLUMNS:
        table[column] = table[column].map(ast.literal_eval)
    lengths = table[list(LITERAL_COLUMNS)].apply(lambda column: column.map(len))
    if not lengths.nunique(axis=1).eq(1).all():
        raise ValueError(f'{path} contains rows with misaligned destination fields.')
    return table

data_root = REPOSITORY_ROOT / config['data']['root']
representation_config = config['data']['representations'][REPRESENTATION]
embedding_path = data_root / config['data']['embedding_directory'] / representation_config['file']
embedding_table = load_embedding_table(embedding_path, representation_config['dimension'])

print(f'Loaded {len(embedding_table):,} Output Area embeddings from {embedding_path.name}.')

Loaded 26,369 Output Area embeddings from london_oa_embeddings_le4mob_dim128_distpres0.1.csv.gz.


In [4]:
class FlowDataset(Dataset):
    def __init__(self, flows: pd.DataFrame, embeddings: pd.DataFrame, distance_column: str):
        self.flows = flows.reset_index(drop=True)
        self.embeddings = embeddings
        self.distance_column = distance_column

    def __len__(self) -> int:
        return len(self.flows)

    def __getitem__(self, index: int):
        row = self.flows.iloc[index]
        origin = str(row['origin'])
        destinations = [str(value) for value in row['destinations']]
        origin_embedding = torch.tensor(self.embeddings.at[origin, 'embedding'], dtype=torch.float32)
        destination_embeddings = torch.tensor(
            self.embeddings.loc[destinations, 'embedding'].tolist(), dtype=torch.float32
        )
        distances = torch.tensor(row[self.distance_column], dtype=torch.float32)
        counts = torch.tensor(row['counts'], dtype=torch.float32)
        return origin_embedding, destination_embeddings, distances, counts, origin

def load_split_datasets(split_id: int):
    flow_root = data_root / config['data']['flow_directory']
    limits = {
        'train': config['debug']['max_train_rows'],
        'val': config['debug']['max_validation_rows'],
        'test': config['debug']['max_test_rows'],
    } if DEBUG else {'train': None, 'val': None, 'test': None}
    frames = {
        name: load_flow_split(flow_root / f'{name}_data_{split_id}.csv.gz', limit=limits[name])
        for name in ('train', 'val', 'test')
    }
    used_ids = set()
    for frame in frames.values():
        used_ids.update(frame['origin'])
        for destinations in frame['destinations']:
            used_ids.update(destinations)
    missing = used_ids - set(embedding_table.index)
    if missing:
        raise ValueError(f'{len(missing)} flow locations have no {REPRESENTATION} embedding.')
    return {
        name: FlowDataset(frame, embedding_table, config['data']['distance_column'])
        for name, frame in frames.items()
    }

preview = load_split_datasets(split_ids[0])
{name: len(dataset) for name, dataset in preview.items()}

{'train': 17139, 'val': 3956, 'test': 5272}

## 4. Model and optimization helpers

The flow-weighted cross-entropy objective matches the `deep-gravity` experiments. DeepGravity consumes normalized distance by default; Bilinear learns an explicit interaction from the two embeddings and ignores distance.

In [5]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

def flow_weighted_cross_entropy(logits: torch.Tensor, counts: torch.Tensor) -> torch.Tensor:
    return -torch.sum(F.log_softmax(logits, dim=-1) * counts)

def build_model():
    dimension = representation_config['dimension']
    common = dict(
        input_dim=dimension,
        hidden_dim=config['model']['hidden_dim'],
        num_hidden_layers=config['model']['num_hidden_layers'],
        dropout_rate=config['model']['dropout'],
    )
    if MODEL_TYPE == 'deepgravity':
        return DeepGravity(
            **common,
            add_distance=config['model']['add_distance'],
            distance_scale=config['data']['distance_scale'],
        )
    if MODEL_TYPE == 'bilinear':
        return Bilinear(
            **common, bilinear_output_dim=config['model']['bilinear_output_dim']
        )
    raise ValueError(f'Unknown model type: {MODEL_TYPE}')

build_model()

DeepGravity(
  (mlp): Sequential(
    (0): Linear(in_features=257, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.5, inplace=False)
    (9): Linear(in_features=128, out_features=128, bias=True)
    (10): ReLU()
    (11): Dropout(p=0.5, inplace=False)
    (12): Linear(in_features=128, out_features=128, bias=True)
    (13): ReLU()
    (14): Dropout(p=0.5, inplace=False)
    (15): Linear(in_features=128, out_features=128, bias=True)
    (16): ReLU()
    (17): Dropout(p=0.5, inplace=False)
    (18): Linear(in_features=128, out_features=128, bias=True)
    (19): ReLU()
    (20): Dropout(p=0.5, inplace=False)
    (21): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [6]:
def train_model(train_dataset: Dataset, validation_dataset: Dataset, seed: int):
    set_seed(seed)
    model = build_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
    train_loader = DataLoader(
        train_dataset, batch_size=config['training']['batch_size'], shuffle=True, drop_last=False
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=config['training']['batch_size'], shuffle=False, drop_last=False
    )

    best_loss = float('inf')
    best_state = None
    best_epoch = -1
    epochs_without_improvement = 0
    global_step = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for origin, destinations, distances, counts, _ in train_loader:
            optimizer.zero_grad()
            logits = model(
                origin.to(device), destinations.to(device), distances=distances.to(device)
            )
            loss = flow_weighted_cross_entropy(logits, counts.to(device))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            global_step += 1

        model.eval()
        validation_loss = 0.0
        with torch.no_grad():
            for origin, destinations, distances, counts, _ in validation_loader:
                logits = model(
                    origin.to(device), destinations.to(device), distances=distances.to(device)
                )
                validation_loss += flow_weighted_cross_entropy(logits, counts.to(device)).item()
        train_loss /= len(train_loader)
        validation_loss /= len(validation_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': validation_loss})
        print(f'epoch={epoch + 1:03d} train={train_loss:.4f} validation={validation_loss:.4f}')

        if validation_loss < best_loss - 1e-5:
            best_loss = validation_loss
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= config['training']['early_stopping_patience']:
                break

    if best_state is None:
        raise RuntimeError('Training did not produce a model state.')
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, global_step

## 5. Evaluation metrics

In [7]:
def common_part_of_commuters(observed: np.ndarray, predicted: np.ndarray) -> float:
    denominator = observed.sum() + predicted.sum()
    return float(2 * np.minimum(observed, predicted).sum() / denominator) if denominator else 0.0

def evaluate_model(model: torch.nn.Module, test_dataset: Dataset):
    model.eval()
    rows = []
    loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    with torch.no_grad():
        for origin, destinations, distances, counts, area_ids in loader:
            probabilities = model.predict_proba(
                origin.to(device), destinations.to(device), distances=distances.to(device)
            ).cpu().numpy().reshape(-1)
            observed = counts.numpy().reshape(-1)
            predicted = probabilities * observed.sum()
            correlation = (
                float(stats.pearsonr(observed, predicted).statistic)
                if len(observed) >= 2 and np.std(observed) > 0 and np.std(predicted) > 0
                else np.nan
            )
            observed_distribution = observed / observed.sum()
            predicted_distribution = predicted / predicted.sum()
            mixture = 0.5 * (observed_distribution + predicted_distribution)
            js_divergence = 0.5 * (
                rel_entr(observed_distribution, mixture).sum()
                + rel_entr(predicted_distribution, mixture).sum()
            )
            jsd = float(np.sqrt(max(0.0, js_divergence)))
            rows.append({
                'oa_id': area_ids[0],
                'true_flows': observed.tolist(),
                'predicted_flows': predicted.tolist(),
                'cpc': common_part_of_commuters(observed, predicted),
                'mae': float(np.mean(np.abs(observed - predicted))),
                'rmse': float(np.sqrt(np.mean((observed - predicted) ** 2))),
                'corr': correlation,
                'jsd': jsd,
            })
    predictions = pd.DataFrame(rows)
    metrics = {name: float(predictions[name].mean()) for name in ('cpc', 'mae', 'rmse', 'corr', 'jsd')}
    return predictions, metrics

## 6. Run the selected experiment

Each split/seed run writes its training history, predictions, and portable downstream checkpoint. The checkpoint records the representation filename and SHA-256 checksum in its metadata but does not duplicate the embedding table. Generated files live under `outputs/` and are ignored by Git.

In [8]:
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
run_directory = (
    REPOSITORY_ROOT / config['output']['root'] / REPRESENTATION / MODEL_TYPE / timestamp
)
run_directory.mkdir(parents=True, exist_ok=True)
resolved_config = deepcopy(config)
resolved_config['selected_representation'] = REPRESENTATION
resolved_config['selected_model_type'] = MODEL_TYPE
resolved_config['resolved_split_ids'] = split_ids
resolved_config['resolved_model_seeds'] = model_seeds
resolved_config['debug_enabled'] = DEBUG
(run_directory / 'config.json').write_text(json.dumps(resolved_config, indent=2))

result_rows = []
embedding_relative_path = embedding_path.relative_to(REPOSITORY_ROOT).as_posix()
embedding_checksum = sha256_file(embedding_path)

for split_id in split_ids:
    print(f'\n=== geographic split {split_id} ===')
    datasets = load_split_datasets(split_id)
    for seed in model_seeds:
        print(f'--- model seed {seed} ---')
        start = time.time()
        model, history, best_epoch, global_step = train_model(
            datasets['train'], datasets['val'], seed
        )
        predictions, metrics = evaluate_model(model, datasets['test'])
        seed_directory = run_directory / f'split_{split_id}' / f'seed_{seed}'
        seed_directory.mkdir(parents=True, exist_ok=True)
        history.to_csv(seed_directory / 'history.csv', index=False)
        predictions.to_csv(seed_directory / 'predictions.csv', index=False)

        model_config = {
            **config['model'],
            'input_dim': representation_config['dimension'],
            'distance_column': config['data']['distance_column'],
            'distance_scale': config['data']['distance_scale'],
        }
        metadata = DownstreamCheckpointMetadata(
            task='commuter_flow',
            model_name=MODEL_TYPE,
            model_config=model_config,
            dataset='london',
            data_split=f'split_{split_id}',
            seed=seed,
            location_embedding_name=REPRESENTATION,
            metrics=metrics,
            extra={
                'embedding_file': embedding_relative_path,
                'embedding_sha256': embedding_checksum,
                'debug': DEBUG,
            },
        )
        save_downstream_checkpoint(
            seed_directory / 'downstream_checkpoint.pt',
            model,
            metadata,
            epoch=best_epoch,
            global_step=global_step,
        )
        result_rows.append({
            'split_id': split_id, 'seed': seed, 'best_epoch': best_epoch,
            'elapsed_seconds': time.time() - start, **metrics,
        })
        print(metrics)

results = pd.DataFrame(result_rows)
results.to_csv(run_directory / 'results.csv', index=False)
run_directory


=== geographic split 0 ===
--- model seed 10 ---
epoch=001 train=21959.5021 validation=3317.2828
epoch=002 train=21735.7959 validation=3310.0252
epoch=003 train=21679.4800 validation=3309.6553
epoch=004 train=21649.4108 validation=3308.2216
epoch=005 train=21607.5258 validation=3300.9728
epoch=006 train=21563.7184 validation=3300.8997
epoch=007 train=21511.2426 validation=3305.2110
epoch=008 train=21490.6983 validation=3274.3191
epoch=009 train=21509.4567 validation=3276.6655
epoch=010 train=21545.5971 validation=3282.6637
epoch=011 train=21583.4266 validation=3268.5498
epoch=012 train=21576.6153 validation=3268.6109
epoch=013 train=21557.9092 validation=3238.7168
epoch=014 train=21498.9815 validation=3216.4863
epoch=015 train=21432.6853 validation=3190.8658
epoch=016 train=21395.5380 validation=3219.6499
epoch=017 train=21409.5912 validation=3167.2219
epoch=018 train=21345.5437 validation=3153.6444
epoch=019 train=21303.3567 validation=3190.3242
epoch=020 train=21252.3113 validation=

PosixPath('/home/xwang/2026/LE4Mob/outputs/commuter_flow/london/le4mob/deepgravity/20260826_164504')

## 7. Aggregate across model seeds and geographic splits

The first table summarizes variation across model seeds within each geographic split. The second gives the mean and standard deviation of split-level means, matching the paper's aggregation protocol.

In [9]:
metric_columns = ['cpc', 'mae', 'rmse', 'corr', 'jsd']
split_summary = results.groupby('split_id')[metric_columns].agg(['mean', 'std'])
split_means = results.groupby('split_id')[metric_columns].mean()
overall_summary = pd.DataFrame({
    'mean': split_means.mean(),
    'std_across_splits': split_means.std(),
})
split_summary.to_csv(run_directory / 'summary_by_split.csv')
overall_summary.to_csv(run_directory / 'summary_overall.csv')
display(split_summary)
display(overall_summary)

cpc           mae          rmse          corr           jsd    
              mean std      mean std      mean std      mean std      mean std
split_id                                                                      
0         0.825017 NaN  0.509535 NaN  0.674375 NaN  0.137306 NaN  0.143712 NaN
1         0.835227 NaN  0.444439 NaN  0.569649 NaN  0.088689 NaN  0.136929 NaN
2         0.827407 NaN  0.467090 NaN  0.619502 NaN  0.089844 NaN  0.147027 NaN
3         0.823666 NaN  0.493374 NaN  0.650183 NaN  0.088915 NaN  0.147377 NaN
4         0.834554 NaN  0.474125 NaN  0.637422 NaN  0.030182 NaN  0.138501 NaN

,mean,std_across_splits
cpc,0.829174,0.005393
mae,0.477713,0.024954
rmse,0.630226,0.039319
corr,0.086987,0.037992
jsd,0.142709,0.004811
